In [ ]:
# Optional: install libraries (uncomment to run)
# pip install --upgrade openai pinecone-client python-dotenv tiktoken


## Tutorial: Combining RAG + CAG (Cache-Augmented Generation)
We’ll add a small cache in front of RAG to speed up repeated queries and save tokens.


### 1) Setup keys and clients


In [ ]:
import os, time, tiktoken
from collections import OrderedDict
from openai import OpenAI
from pinecone import Pinecone

OPENAI_API_KEY = OPENROUTER_API_KEY
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
assert OPENAI_API_KEY and PINECONE_API_KEY, "Missing API keys"

client = OpenAI(api_key=OPENAI_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)

EMBED_MODEL = "openai/text-embedding-3-small"
MODEL = "gpt-4o-mini"
enc = tiktoken.get_encoding("cl100k_base")


### 2) Reuse existing index or create a tiny one


In [ ]:
from pinecone import ServerlessSpec

In [ ]:
def embed(text: str):
    r = client.embeddings.create(model=EMBED_MODEL, input=text)
    return r.data[0].embedding

INDEX_NAME = "ragcag-demo"
try:
    index = pc.Index(INDEX_NAME)
    _ = index.describe_index_stats()
except Exception:
    tiny = [
        {"id": "d1", "text": "RAG retrieves context to ground LLMs."},
        {"id": "d2", "text": "Caching previous answers can speed up repeated queries."},
    ]
    vecs = [{"id": d["id"], "values": embed(d["text"]), "metadata": {"text": d["text"]}} for d in tiny]
    DIM = len(vecs[0]["values"])
    existing = [idx.name for idx in pc.list_indexes()]
    if INDEX_NAME not in existing:
        pc.create_index(
            name=INDEX_NAME,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
    index = pc.Index(INDEX_NAME)
    index.upsert(vectors=vecs, namespace="ns1")

"ready"


### 3) Tiny LRU+TTL cache


In [ ]:
class LruTtlCache:
    def __init__(self, capacity: int = 16, ttl_seconds: int = 600):
        self.capacity = capacity
        self.ttl = ttl_seconds
        self.data = OrderedDict()  # key -> (value, expiry)

    def get(self, key: str):
        now = time.time()
        if key in self.data:
            value, expiry = self.data.pop(key)
            if expiry > now:
                self.data[key] = (value, expiry)
                return value
        return None

    def set(self, key: str, value):
        now = time.time()
        expiry = now + self.ttl
        if key in self.data:
            self.data.pop(key)
        elif len(self.data) >= self.capacity:
            self.data.popitem(last=False)  # evict LRU
        self.data[key] = (value, expiry)

cache = LruTtlCache(capacity=8, ttl_seconds=300)


### 4) RAG primitives: retrieve and answer


In [ ]:
def retrieve(query: str, top_k: int = 3):
    qv = embed(query)
    res = index.query(vector=qv, top_k=top_k, include_metadata=True)
    return [m["metadata"]["text"] for m in res["matches"]]

def answer_with_context(question: str, context: str):
    prompt = f"Use only the context to answer.\n\nContext:\n{context}\n\nQuestion: {question}"
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "If unsure from context, say you don't know."},
            {"role": "user", "content": prompt},
        ],
    )
    return resp.choices[0].message.content


### 5) Unified flow: cache → retrieval → generation
Cache hit returns instantly; cache miss performs RAG then stores the answer.


In [ ]:
def rag_cag_answer(question: str):
    cached = cache.get(question)
    if cached is not None:
        return "[cache hit] " + cached
    snippets = retrieve(question, top_k=3)
    context = "\n\n".join(snippets)
    ans = answer_with_context(question, context)
    cache.set(question, ans)
    return "[cache miss] " + ans

q = "How can caching help with repeated RAG queries?"
print(rag_cag_answer(q))  # miss first
print(rag_cag_answer(q))  # hit second


In [ ]:
print(rag_cag_answer(q))